# Görev #14 — Cihaz-içi VRAM / Latency Vitrini (HFP-hibrit vs saf KV-cache)

**Amaç.** Vizyonun *sayısal grafiği*: bağlam uzadıkça (4k → 128k) belleğin ve token
başına gecikmenin nasıl büyüdüğünü, HFP-hibrit ile saf Qwen2.5-1.5B (tam KV-cache)
için yan yana ölç.

## ÖN-KAYITLI DÜRÜSTLÜK NOTLARI (koşudan önce yazıldı — AGENTS.md kural #1)

1. **Bu ölçüm YAPISALDIR, checkpoint'ten bağımsızdır.** VRAM ve latency yalnızca
   tensör *şekillerine*, dtype'a ve *compute desenine* bağlıdır; eğitilmiş
   ağırlık değerlerine değil. Bu yüzden benchmark, eğitilmemiş bir graft ile
   birebir aynı eğriyi verir ve eğitim gerektirmez. **Kalite** (PPL 1.11×,
   needle 12/12) ayrı bir olgudur ve RESULTS §22'de kanıtlanmıştır. İki iddia
   ayrı tutulur.

2. **Tasarruf graft yoğunluğuyla ORANTILIDIR — abartma yok.** Referans reçete
   28 katmandan **6**'sını O(1) belleğe çevirir. KV-cache yalnızca kalan 22
   full-attention katmanda birikir. Beklenen KV tasarrufu ≈ 6/28 ≈ **%21**.
   Bu gerçek ama mütevazıdır. Grafik bunu olduğu gibi gösterir; "O(1) modele
   yaklaşım" ancak graft yoğunluğu artırılarak gelir ve bunun bir PPL bedeli
   vardır (§22a uçurumu). Sunum bu Pareto'yu saklamaz.

3. **Adil kıyas.** Aynı ağırlıklar, aynı dtype (fp16), aynı prefill chunk'ı,
   aynı decode token sayısı, aynı GPU. Tek değişken: graft var/yok.

4. **O(1) imzası.** Graft'lı katmanların taşıdığı durum (M, z) bağlam
   uzunluğundan BAĞIMSIZDIR (sabit ~MB). Saf KV-cache ise token sayısıyla
   doğrusal büyür. Asıl vitrin bu kontrasttır: düz çizgi vs artan çizgi.

**Kriterler.** (a) HFP-hibrit tepe VRAM'i her ölçülen L'de baseline'dan DÜŞÜK
olmalı ve fark L ile büyümeli (KV tasarrufu birikir). (b) Graft state boyutu L
ile sabit kalmalı. (c) Beklenen KV tasarruf oranı analitik ~%21'e yakın çıkmalı;
sapma varsa implementasyon (cache gerçekten atılıyor mu) sorgulanır.

**Ortam.** Colab/Kaggle T4 (16 GB) yeter. Eğitim yok → dakikalar mertebesinde.


In [ ]:
# --- 1. KURULUM (egitim yok; yalniz model + repo) ---
import os, subprocess, sys, glob, time, json, math
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_XET'] = '1'          # Xet CDN 403/stall bypass (bkz. ana notebook)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

import torch
assert torch.cuda.is_available(), 'GPU SECILI DEGIL! Runtime > Change runtime type > T4 GPU.'
_cap = torch.cuda.get_device_capability(0); _arch = f'sm_{_cap[0]}{_cap[1]}'
assert _arch in torch.cuda.get_arch_list(), (
    f'GPU mimarisi {_arch} bu torch derlemesinde YOK -> T4 secin (P100 desteklenmiyor).')
DEV = 'cuda'
print('GPU:', torch.cuda.get_device_name(0), '| toplam VRAM',
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers>=4.46', 'accelerate'], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)

# Ciktilar (grafik + CSV) icin dizin — Drive varsa oraya (kalici), yoksa repoya.
OUT_DIR = os.path.join(REPO, 'docs', 'assets')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/hfp_bench'
except Exception as e:
    print(f'Drive yok ({type(e).__name__}) -> ciktilar repoya yazilir.')
os.makedirs(OUT_DIR, exist_ok=True)
print('Cikti dizini:', OUT_DIR)

In [ ]:
# --- 2. BENCHMARK PARAMETRELERI ---
# Referans rece te = §22a: 6-katman graft, exp decay, additive yazim (triangular-solve
# YOK -> fp16 guvenli). Katmanlar Run 7/8/9 ile ayni.
GRAFT_LAYERS   = [3, 7, 11, 15, 19, 23]
DECAY_MODE     = 'exp'
WRITE_RULE     = 'additive'
DTYPE          = torch.float16          # cihaz-ici gercekci dtype; 128k'yi T4'e sigdirir
PREFILL_CHUNK  = 512                    # prefill'i parcala -> aktivasyon bellegi sinirli
DECODE_TOKENS  = 32                     # token-basi latency icin decode adimi
CONTEXT_LENGTHS = [4096, 8192, 16384, 32768, 65536, 131072]

MODEL_ID = 'Qwen/Qwen2.5-1.5B'
print(f'Recete: {len(GRAFT_LAYERS)}/28 katman graft {GRAFT_LAYERS} | {DECAY_MODE}+{WRITE_RULE}')
print(f'dtype={DTYPE} chunk={PREFILL_CHUNK} decode={DECODE_TOKENS} '
      f'L={CONTEXT_LENGTHS}')

In [ ]:
# --- 3. MODEL YUKLEME YARDIMCISI (Kaggle Input / Drive / HF token) ---
from transformers import AutoModelForCausalLM, AutoTokenizer

def _find_local_model():
    roots = ['/kaggle/input', OUT_DIR, BASE, '/content/drive/MyDrive', '/content']
    seen = set()
    for root in roots:
        if not root or root in seen or not os.path.isdir(root): continue
        seen.add(root)
        for cfg in glob.glob(f'{root}/**/config.json', recursive=True):
            d = os.path.dirname(cfg)
            if glob.glob(f'{d}/*.safetensors'): return d
    return None

def load_qwen():
    src = _find_local_model()
    if src:
        print('Model yerelden (indirme yok):', src)
    else:
        _tk = os.environ.get('HF_TOKEN')
        try:
            from google.colab import userdata
            _tk = _tk or userdata.get('HF_TOKEN')
        except Exception: pass
        assert _tk, ('Model yerelde yok ve HF_TOKEN yok. Colab: Secrets > HF_TOKEN ekle; '
                     'Kaggle: Add Input ile Qwen2.5-1.5B ekle.')
        from huggingface_hub import snapshot_download
        src = f'{BASE}/qwen_model'
        snapshot_download(repo_id=MODEL_ID, local_dir=src, token=_tk,
                          allow_patterns=['*.json','*.txt','tokenizer.model',
                                          '*.safetensors','*.safetensors.index.json'],
                          max_workers=2)
    tokz = AutoTokenizer.from_pretrained(src)
    # sdpa (mem-efficient): her iki modelde ayni -> adil; 128k'yi T4'e sigdirir.
    mdl = AutoModelForCausalLM.from_pretrained(src, torch_dtype=DTYPE,
                                               attn_implementation='sdpa').to(DEV).eval()
    return mdl, tokz, src

# Model config'inden analitik KV-cache buyuklugu (dogrulama icin)
def kv_bytes_per_token(cfg, n_layers, dtype=DTYPE):
    kv_heads = getattr(cfg, 'num_key_value_heads', cfg.num_attention_heads)
    hd = getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads
    elem = torch.tensor([], dtype=dtype).element_size()
    return 2 * kv_heads * hd * n_layers * elem      # K+V
print('yardimcilar hazir')

In [ ]:
# --- 4. OLCUM CEKIRDEGI ---
from transformers import DynamicCache
from hfp.models.grafting import (graft_llama, GraftConfig, set_graft_mode,
                                  enable_streaming, reset_streaming, HFPGraftAttention)

@torch.no_grad()
def measure(model, L, is_hybrid):
    """L-token prefill (chunk'li) + DECODE_TOKENS decode. Doner: dict(metrikler) | None(OOM)."""
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    ids = torch.randint(0, 30000, (1, L), device=DEV)
    try:
        if is_hybrid:
            set_graft_mode(model, 'student'); enable_streaming(model, True); reset_streaming(model)
        cache = DynamicCache()
        torch.cuda.synchronize(); t0 = time.time()
        out = None
        for s in range(0, L, PREFILL_CHUNK):
            out = model(ids[:, s:s+PREFILL_CHUNK], past_key_values=cache, use_cache=True)
            cache = out.past_key_values
        torch.cuda.synchronize(); prefill_s = time.time() - t0
        # decode: son token'dan itibaren DECODE_TOKENS adim
        last = out.logits[:, -1:].argmax(-1)
        torch.cuda.synchronize(); t1 = time.time()
        for _ in range(DECODE_TOKENS):
            out = model(last, past_key_values=cache, use_cache=True)
            cache = out.past_key_values
            last = out.logits[:, -1:].argmax(-1)
        torch.cuda.synchronize(); decode_s = time.time() - t1
        peak = torch.cuda.max_memory_allocated()
        # graft state boyutu (O(1) imzasi) — yalniz hibritte
        state_mb = 0.0
        if is_hybrid:
            for m in model.modules():
                if isinstance(m, HFPGraftAttention) and m._stream_state is not None:
                    M, z = m._stream_state[0], m._stream_state[1]
                    state_mb += (M.numel()*M.element_size() + z.numel()*z.element_size())/1e6
            enable_streaming(model, False)
        # HF cache'teki gercek katman sayisi (hibritte 22 olmali)
        cached_layers = -1
        try:
            kc = getattr(cache, 'key_cache', None)
            if kc is None and hasattr(cache, 'layers'):
                kc = [getattr(l, 'keys', None) for l in cache.layers]
            if kc is not None:
                cached_layers = sum(1 for lk in kc if lk is not None and getattr(lk, 'numel', lambda: 0)() > 0)
        except Exception:
            cached_layers = -1
        return dict(L=L, peak_gb=peak/1e9, prefill_s=prefill_s,
                    decode_ms_tok=1000*decode_s/DECODE_TOKENS,
                    state_mb=round(state_mb, 2), cached_layers=cached_layers)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f'   L={L}: OOM (bu uzunluk bu GPU\'ya sigmadi) -> durduruldu')
        return None

print('olcum cekirdegi hazir')

In [ ]:
# --- 5. SWEEP: once BASELINE (saf Qwen), sonra HYBRID (ayni agirliklar, graft'li) ---
# Bellek confound'unu onlemek icin tek seferde tek model resident.

def sweep(is_hybrid, tag):
    model, tok, src = load_qwen()
    n_layers_cached = model.config.num_hidden_layers
    if is_hybrid:
        cfg = GraftConfig(decay_mode=DECAY_MODE, write_rule=WRITE_RULE, key_feature_map='dpfp')
        graft_llama(model, cfg, layers=GRAFT_LAYERS)
        set_graft_mode(model, 'student')
        n_layers_cached = model.config.num_hidden_layers - len(GRAFT_LAYERS)
    kv_bt = kv_bytes_per_token(model.config, n_layers_cached)
    print(f'\n=== {tag} === (KV-tutan katman: {n_layers_cached}, '
          f'analitik {kv_bt} B/token)')
    # warmup (kernel derleme / autotune disari)
    _ = measure(model, 1024, is_hybrid)
    rows = []
    for L in CONTEXT_LENGTHS:
        r = measure(model, L, is_hybrid)
        if r is None: break
        r['tag'] = tag; r['kv_analytic_gb'] = kv_bt * L / 1e9
        rows.append(r)
        print(f'   L={L:>7}: VRAM {r["peak_gb"]:.2f} GB | prefill {r["prefill_s"]:.2f}s | '
              f'decode {r["decode_ms_tok"]:.1f} ms/tok | state {r["state_mb"]:.1f} MB | '
              f'cache_layers={r["cached_layers"]}')
    del model; torch.cuda.empty_cache()
    return rows

base_rows   = sweep(False, 'baseline')
hybrid_rows = sweep(True,  'hfp_hybrid')

ALL = base_rows + hybrid_rows
with open(os.path.join(OUT_DIR, 'vram_latency_bench.json'), 'w') as f:
    json.dump(dict(model=MODEL_ID, dtype=str(DTYPE), graft_layers=GRAFT_LAYERS,
                   decode_tokens=DECODE_TOKENS, prefill_chunk=PREFILL_CHUNK,
                   gpu=torch.cuda.get_device_name(0), rows=ALL), f, indent=2)
print('\nJSON yazildi:', os.path.join(OUT_DIR, 'vram_latency_bench.json'))

In [ ]:
# --- 6. GRAFIK + DURUST YORUM ---
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

def xy(rows, k): return [r['L'] for r in rows], [r[k] for r in rows]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
# (a) Tepe VRAM
bx, by = xy(base_rows, 'peak_gb'); hx, hy = xy(hybrid_rows, 'peak_gb')
ax[0].plot(bx, by, 'o-', label='saf Qwen (28-kat KV)', color='#c0392b')
ax[0].plot(hx, hy, 's-', label='HFP-hibrit (6-kat O(1))', color='#2471a3')
ax[0].set_title('Tepe VRAM'); ax[0].set_ylabel('GB')
# (b) decode latency
bx, by = xy(base_rows, 'decode_ms_tok'); hx, hy = xy(hybrid_rows, 'decode_ms_tok')
ax[1].plot(bx, by, 'o-', color='#c0392b', label='saf Qwen'); ax[1].plot(hx, hy, 's-', color='#2471a3', label='HFP-hibrit')
ax[1].set_title('Decode gecikmesi'); ax[1].set_ylabel('ms / token')
# (c) O(1) imzasi: graft state (duz) vs analitik KV (artan)
hx, hy = xy(hybrid_rows, 'state_mb')
kx = [r['L'] for r in base_rows]; ky = [r['kv_analytic_gb']*1000 for r in base_rows]
ax[2].plot(kx, ky, 'o-', label='saf KV-cache (analitik)', color='#c0392b')
ax[2].plot(hx, hy, 's-', label='HFP graft state', color='#2471a3')
ax[2].set_title('Bellek imzasi: artan vs sabit'); ax[2].set_ylabel('MB'); ax[2].set_yscale('log')
for a in ax:
    a.set_xlabel('baglam (token)'); a.set_xscale('log'); a.grid(alpha=.3)
    if a.get_legend_handles_labels()[1]: a.legend(fontsize=8)
fig.suptitle('HFP-hibrit vs saf KV-cache — Qwen2.5-1.5B, T4, fp16 (6/28 katman graft)', y=1.02)
fig.tight_layout()
png = os.path.join(OUT_DIR, 'vram_latency_bench.png')
fig.savefig(png, dpi=130, bbox_inches='tight'); print('Grafik:', png)

# --- DURUST YORUM (olculen sayilardan) ---
def at(rows, L, k):
    for r in rows:
        if r['L'] == L: return r[k]
    return None
print('\n--- OZET (dogrulama) ---')
common = [r['L'] for r in hybrid_rows if any(b['L']==r['L'] for b in base_rows)]
for L in common:
    vb, vh = at(base_rows,L,'peak_gb'), at(hybrid_rows,L,'peak_gb')
    save = 100*(vb-vh)/vb
    print(f'L={L:>7}: VRAM {vb:.2f} -> {vh:.2f} GB  (tasarruf %{save:.1f}) | '
          f'decode {at(base_rows,L,"decode_ms_tok"):.1f} -> {at(hybrid_rows,L,"decode_ms_tok"):.1f} ms/tok')
Lmax = common[-1]
exp_save = 100*len(GRAFT_LAYERS)/28
print(f'\nBeklenen KV tasarrufu (analitik) ≈ %{exp_save:.0f} (6/28 katman).')
print(f'En uzun L={Lmax}: graft state {at(hybrid_rows,Lmax,"state_mb"):.1f} MB (SABIT), '
      f'saf KV analitik {at(base_rows,Lmax,"kv_analytic_gb")*1000:.0f} MB (L ile artar).')
_cl = at(hybrid_rows,Lmax,"cached_layers"); _expected = 28 - len(GRAFT_LAYERS)
if _cl == _expected:
    print(f'HF cache katman sayisi hibritte {_cl} (=28-{len(GRAFT_LAYERS)}): graft KV yazmiyor -> DOGRUDAN DOGRULANDI.')
elif _cl == -1:
    print('HF cache katman sayisi API''den okunamadi -> DOLAYLI kanit: olculen VRAM tasarrufu analitik KV tahminiyle ortusuyor.')
else:
    print(f'UYARI: cache katman {_cl}, beklenen {_expected} DEGIL -> incelenmeli.')
print('\nDURUST CERCEVE: tasarruf 6/28 graft yogunluguyla ORANTILI; daha yuksek '
      'yogunluk O(1) asimptotuna yaklastirir ama PPL bedeli vardir (RESULTS §22a). '
      'Bu grafik mevcut recetenin Pareto noktasini gosterir, tavani degil.')